In [1]:
using CSV
using DataFrames
using Dates
using Distributions
using Serialization
using LinearAlgebra: diagm

In [2]:
projectbasepath = "../../covid-resource-allocation/"
include(joinpath(projectbasepath, "src/processing/LocalDataCommon.jl"));
include(joinpath(projectbasepath, "src/processing/GeographicData.jl"));

In [3]:
data_dir = "../data/";
outputdatapath = "../data/";

In [4]:
forecast_data = DataFrame(CSV.File("../data/shortterm-2021-03-30.csv"));

In [5]:
real_data = DataFrame(CSV.File("../data/jhhs_realdata_2021_03_30.csv"));

In [6]:
SCENARIOS = [:shortterm];
BEDTYPES  = [:allbeds];

In [7]:
los_dist = (
    allbeds = Gamma(2.244, 4.4988),
    ward = Gamma(2.601, 3.8046),
    icu = Gamma(1.77595, 5.9512),
);

In [8]:
start_date = minimum(forecast_data.date);
end_date   = maximum(forecast_data.date);
date_range = collect(start_date : Day(1) : end_date);
@show start_date, end_date;

(start_date, end_date) = (Date("2021-03-29"), Date("2021-04-11"))


In [9]:
hospitals = ["JHH", "SH", "BMC", "HCGH", "SMH"];
N = length(hospitals);

In [10]:
adj = BitArray(ones(N,N) - diagm(ones(N)));

In [11]:
dist_matrix = diagm(fill(Inf, N));

In [12]:
capacity_names_full = ["Base Capacity", "Ramp-Up Capacity", "Surge Capacity", "Surge+ Capacity", "Maximum Capacity"];
capacity_names_abbrev = [:baseline, :rampup, :surge, :surgeplus, :max];

In [13]:
function load_capacity_jhhs(hospitals, bedtype, capacity_levels=[:baseline])
    beds_data = DataFrame(CSV.File(joinpath(data_dir, "jhhs_beds.csv")))

    beds_dict = Dict(row.hospital => Dict(
        "icu_covid_baseline" => row.icu_covid_baseline,
        "ward_covid_baseline" => row.ward_covid_baseline,
        "allbeds_covid_baseline" => row.icu_covid_baseline + row.ward_covid_baseline,
        "icu_covid_rampup" => row.icu_covid_rampup,
        "ward_covid_rampup" => row.ward_covid_rampup,
        "allbeds_covid_rampup" => row.icu_covid_rampup + row.ward_covid_rampup,
        "icu_covid_surge" => row.icu_covid_surge,
        "ward_covid_surge" => row.ward_covid_surge,
        "allbeds_covid_surge" => row.icu_covid_surge + row.ward_covid_surge,
        "icu_covid_surgeplus" => row.icu_covid_surgeplus,
        "ward_covid_surgeplus" => row.ward_covid_surgeplus,
        "allbeds_covid_surgeplus" => row.icu_covid_surgeplus + row.ward_covid_surgeplus,
        "icu_covid_max" => row.icu_covid_max,
        "ward_covid_max" => row.ward_covid_max,
        "allbeds_covid_max" => row.icu_covid_max + row.ward_covid_max,
    ) for row in eachrow(beds_data))

    if capacity_levels isa Symbol
        capacity = [beds_dict[h]["$(bedtype)_covid_$(capacity_levels)"] for h in hospitals]
    elseif capacity_levels isa AbstractArray
        capacity = hcat([[beds_dict[h]["$(bedtype)_covid_$(l)"] for h in hospitals] for l in capacity_levels]...)
    else
        error("Invalid capacity_levels")
    end

    return capacity
end;

In [14]:
function estimate_active(_initial::Array{<:Real,1}, _admitted::Array{<:Real,2}, los_dist::Distribution)
    N, T = size(_admitted)
    _L = 1.0 .- cdf.(los_dist, 0:T)
    _discharged = [_initial[i] * pdf(los_dist, t) for i in 1:N, t in 1:T]
    _active = [(
        _initial[i]
        - sum(_discharged[i,1:t])
        + sum(_L[t-t₁+1] * _admitted[i,t₁] for t₁ in 1:t)
    ) for i in 1:N, t in 1:T]
    return _active
end;

In [15]:
function estimate_admitted(active::Array{<:Real,1}, los_dist)
    T = length(active)
    
    initial = active[1]
    discharged = initial .* (pdf.(los_dist, 0:T-1))

    L = 1.0 .- cdf.(los_dist, 0:T)

    A = [(t′ ≤ t) ? L[t-t′+1] : 0 for t in 1:T, t′ in 1:T]
    b = [active[t] - (initial - sum(discharged[1:t])) for t in 1:T]
    admitted = A \ b
    
    return admitted
end;

function estimate_admitted(active::Array{<:Real,2}, los_dist)
    admitted = Array{Float64,2}(undef, size(active)...)
    for i in 1:size(active,1)
        admitted[i,:] = estimate_admitted(active[i,:], los_dist)
    end
    return admitted
end;

In [16]:
function load_data_jhhs(scenario, bedtype)
    
    forecast_active_dict = Dict((row.hospital, row.date) => row.occupancy_total for row in eachrow(forecast_data))
    forecast_active = [forecast_active_dict[(h,d)] for h in hospitals, d in date_range]
    
    forecast_admitted_dict = Dict((row.hospital, row.date) => row.admissions_total for row in eachrow(forecast_data))
    forecast_admitted = [forecast_admitted_dict[(h,d)] for h in hospitals, d in date_range]

    day0 = start_date - Day(1)
    initial_data = filter(row -> row.date == day0, real_data)
    initial_dict = Dict(row.hospital => row.active_total for row in eachrow(initial_data))
    initial = [initial_dict[h] for h in hospitals]
    
    forecast_admitted_uncertainty = 0.15 .* forecast_admitted
    
    beds = load_capacity_jhhs(hospitals, bedtype, :baseline)
    capacity = load_capacity_jhhs(hospitals, bedtype, capacity_names_abbrev)

    data = (
        scenario = scenario,
        bedtype = bedtype,

        los_dist = los_dist[bedtype],

        initial = initial,
        active = forecast_active,
        admitted = forecast_admitted,
        admitted_uncertainty = forecast_admitted_uncertainty,

        beds = beds,
        capacity = capacity,
    )

    return data
end;

In [17]:
maindata = Dict()
for scenario in SCENARIOS, bedtype in BEDTYPES
    maindata[(scenario,bedtype)] = load_data_jhhs(scenario, bedtype)
end

In [18]:
hospital_positions = Dict(
    "JHH"  => (lat = 39.2961773, long = -76.5939447),
    "SMH"  => (lat = 38.9364687, long = -77.1091435),
    "HCGH" => (lat = 39.2136187, long = -76.885917),
    "BMC"  => (lat = 39.290101,  long = -76.5468383),
    "SH"   => (lat = 38.9973285, long = -77.1105309),
);

In [19]:
completedata = (
    location_names = hospitals,
    location_names_short = hospitals,
    start_date = start_date,
    end_date = end_date,
    counties = nothing,
    states = nothing,
    dist_matrix = dist_matrix,
    locations_latlong = hospital_positions,
    capacity_names = capacity_names_full,
    casesdata = maindata,
);

In [20]:
serialize(joinpath(outputdatapath, "data_jhhs_shortterm.jlser"), completedata);